In [ ]:
import sys
sys.path.append('..')

import os
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from PIL import Image

from diffusers.utils import load_video, export_to_video
from diffusers import AutoencoderKLWan, FlowMatchEulerDiscreteScheduler, UniPCMultistepScheduler
from diffusers.video_processor import VideoProcessor
from transformers import AutoTokenizer, UMT5EncoderModel, T5TokenizerFast
from denku import show_images

from wan_continuous_transformer import WanTransformer3DModel
from wan_continuous_pipeline import WanContinuousVideoPipeline

%load_ext autoreload
%autoreload 2

In [ ]:
model_id = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"

tokenizer = T5TokenizerFast.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = UMT5EncoderModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=torch.bfloat16)
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(model_id, subfolder="scheduler")

In [ ]:
transformer = WanTransformer3DModel.from_pretrained(model_id, subfolder="transformer", torch_dtype=torch.bfloat16)

In [ ]:
pipe = WanContinuousVideoPipeline.from_pretrained(
    pretrained_model_name_or_path=model_id,
    tokenizer=tokenizer, 
    text_encoder=text_encoder,
    transformer=transformer,
    vae=vae, 
    scheduler=scheduler,
)
# pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()

In [ ]:
lora_path = "TheDenk/wan2.2-video-continuation"

pipe.transformer.load_lora_adapter(
    lora_path,
    weight_name="pytorch_lora_weights.safetensors",
    adapter_name="video_continuation",
    prefix=None,
)
pipe.set_adapters("video_continuation", adapter_weights=1.0)

# pipeline.fuse_lora(adapter_names=["video_continuation"], lora_scale=1.0)
# pipeline.unload_lora_weights()

In [ ]:
img_h = 480 # 704 480
img_w = 832 # 1280 832
input_frames_count = 24 # 33
output_frames_count = 113 - input_frames_count  # 121 81 49
print(input_frames_count, output_frames_count)

# video_path = '../resources/dog.mp4'
# video_path = '../resources/escalator.mp4'
# video_path = '../resources/bubble.mp4'
video_path = '../resources/ship.mp4'

video_frames = load_video(video_path)[:input_frames_count]
show_images(video_frames[::8], figsize=(16, 8))

In [ ]:
# prompt = "In a cozy kitchen, a golden retriever wearing a white chef's hat and a blue apron stands at the table, holding a sharp kitchen knife and skillfully slicing fresh tomatoes. Its tail sways gently, and its gaze is focused and gentle. There are already several neatly arranged tomatoes on the wooden chopping board in front of me. The kitchen has soft lighting, with various kitchen utensils hanging on the walls and several pots of green plants placed on the windowsill."
# prompt = "Wide shot，The video shows a person in a red outfit standing on an escalator, facing away from the camera. The escalator is moving upwards, and the person appears to be stationary. The surroundings are dimly lit with reflective surfaces that create a mirrored effect, giving the impression of multiple identical figures ascending simultaneously."
# prompt = "Close-up shot with soft lighting, focusing sharply on the lower half of a young woman's face. Her lips are slightly parted as she blows an enormous bubblegum bubble. The bubble is semi-transparent, shimmering gently under the light, and surprisingly contains a miniature aquarium inside, where two orange-and-white goldfish slowly swim, their fins delicately fluttering as if in an aquatic universe. The background is a pure light blue color."
prompt = "Watercolor style, the wet suminagashi inks slowly spread into the shape of an island on the paper, with the edges continuously blending into delicate textural variations. A tiny paper boat floats in the direction of the water flow towards the still-wet areas, creating subtle ripples around it. Centered composition with soft natural light pouring in from the side, revealing subtle color gradations and a sense of movement. "

negative_prompt = "bad quality, worst quality"

In [ ]:
num_cycles = 1

out = [x.resize((img_w, img_h)) for x in video_frames]
for num_cycle in range(num_cycles):
    output = pipe(
        previous_video=out[-input_frames_count:] if num_cycle > 0 else video_frames,
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=img_h,
        width=img_w,
        num_frames=output_frames_count,
        guidance_scale=5,
        generator=torch.Generator(device="cuda").manual_seed(42),
        output_type="pil",
        
        teacache_treshold=0.4,
    ).frames[0]
    out += output

    export_to_video(out, f"continuous_output_{num_cycle}_iteration.mp4", fps=24)